# 02 Azure Cloud dla QA - Azure DevOps: CI/CD i Test Plans

_Kamil Bartocha_ | wersja 1.0

## Rozklad jazdy

1. 🔹 Azure DevOps - przeglad platformy
2. 🔹 YAML Pipeline - podstawy skladni
3. 🔹 Pytest w pipeline - uruchamianie i raportowanie
4. 🔹 Multi-stage pipeline dla QA
5. 🔹 Azure DevOps REST API w Pythonie

---

## 1. 🔹 Azure DevOps - przeglad platformy

Azure DevOps to platforma Microsoft do zarzadzania cykliem zycia
oprogramowania. Sklada sie z pieciu modulow (services):

| Modul         | Opis                                              |
|---------------|---------------------------------------------------|
| **Boards**    | Zarzadzanie zadaniami (backlog, sprinty, Kanban)  |
| **Repos**     | Repozytoria Git (podobne do GitHub)               |
| **Pipelines** | CI/CD - automatyczne budowanie, testowanie, deploy|
| **Test Plans**| Zarzadzanie testami, test runs, raportowanie      |
| **Artifacts** | Pakiety NuGet, npm, PyPI, Maven                   |

**Struktura organizacji:**
```
Organization  (np. dev.azure.com/mycompany)
 └── Project  (np. checkout-service)
      ├── Repos     - repozytoria kodu
      ├── Pipelines - definicje CI/CD
      ├── Test Plans - plany i wyniki testow
      └── Artifacts - pakiety
```

**Azure DevOps vs GitHub Actions:**

| Aspekt               | Azure DevOps Pipelines    | GitHub Actions            |
|----------------------|---------------------------|---------------------------|
| Integracja           | Azure, Microsoft          | GitHub, ekosystem OSS     |
| Test Plans           | wbudowane                 | brak - zewnetrzne narzedzie|
| Self-hosted agents   | rozbudowane               | podstawowe                |
| Approval gates       | natywne w Environments    | Environments (ograniczone)|
| Kiedy uzywac         | korporacje, Azure-native  | projekty GitHub, OSS      |

> 💡 Dla QA: Azure Test Plans + Pipelines to kompletne rozwiazanie -
> wyniki z Pytest automatycznie trafiaja do Test Plans bez integracji
> z zewnetrznym narzedziem (Jira, TestRail).

In [ ]:
import os
import requests
from base64 import b64encode


class AdoClient:
    """Minimal Azure DevOps REST API client using PAT authentication."""

    API_VERSION = "7.1"

    def __init__(self, org, project, pat):
        self.base = f"https://dev.azure.com/{org}/{project}/_apis"
        self.auth = ("", pat)

    def get(self, resource, **params):
        params["api-version"] = self.API_VERSION
        resp = requests.get(f"{self.base}/{resource}", auth=self.auth, params=params)
        resp.raise_for_status()
        return resp.json()

    def post(self, resource, body):
        params = {"api-version": self.API_VERSION}
        resp = requests.post(
            f"{self.base}/{resource}", auth=self.auth,
            params=params, json=body
        )
        resp.raise_for_status()
        return resp.json()


# Konfiguracja - uzupelnij zmienne srodowiskowe lub wpisz wartosci
ORG = os.environ.get("ADO_ORG", "my-organization")
PROJECT = os.environ.get("ADO_PROJECT", "my-project")
PAT = os.environ.get("ADO_PAT", "")

client = AdoClient(ORG, PROJECT, PAT)

# Pobieramy info o projekcie
project_info = client.get("../..".join([""]) + f"https://dev.azure.com/{ORG}/_apis/projects/{PROJECT}".split("_apis")[1], )
# Uproszczony sposob - bezposrednie zapytanie
resp = requests.get(
    f"https://dev.azure.com/{ORG}/_apis/projects/{PROJECT}",
    auth=("", PAT),
    params={"api-version": "7.1"}
)
if resp.status_code == 200:
    info = resp.json()
    print(f"Projekt:      {info['name']}")
    print(f"ID:           {info['id']}")
    print(f"Stan:         {info['state']}")
    print(f"Widocznosc:   {info['visibility']}")
else:
    print(f"Blad: {resp.status_code} - uzupelnij ADO_ORG, ADO_PROJECT, ADO_PAT")

---

### 🐍 Cwiczenia - Azure DevOps przeglad

**1.** Uzupelnij konfiguracje `AdoClient` swoimi danymi
(ORG, PROJECT, PAT) i wypisz nazwe oraz ID projektu.

**2.** Korzystajac z REST API pobierz liste repozytoriow
w projekcie (`git/repositories`) i wypisz ich nazwy.

**3.** Pobierz liste potoków (pipelines) w projekcie
i wypisz ich nazwy oraz ID w formie tabeli. *(Trudniejsze)*

In [ ]:
# Cwiczenie 1 - info o projekcie
ORG = ...
PROJECT = ...
PAT = ...

resp = requests.get(
    f"https://dev.azure.com/{ORG}/_apis/projects/{PROJECT}",
    auth=("", PAT),
    params={"api-version": "7.1"}
)
resp.raise_for_status()
info = resp.json()

print(f"Projekt: {info['name']}")
print(f"ID:      {info['id']}")

In [ ]:
# Cwiczenie 2 - lista repozytoriow
# hint: endpoint to 'git/repositories', odpowiedz ma klucz 'value'
client = AdoClient(ORG, PROJECT, PAT)
repos_data = client.get(...)
repos = repos_data["value"]

print(f"Repozytoria ({len(repos)}):")
for repo in repos:
    print(f"  {repo['name']}")

In [ ]:
# Cwiczenie 3 - lista pipeline'ow *(Trudniejsze)*
# hint: endpoint to 'pipelines', pola: 'id', 'name', 'folder'
pipelines_data = client.get(...)
pipelines = pipelines_data["value"]

print(f"{'ID':<8} {'Folder':<30} Nazwa")
print("-" * 60)
for p in pipelines:
    print(f"{p['id']:<8} {p.get('folder', '\\'):<30} {p['name']}")

---

## 2. 🔹 YAML Pipeline - podstawy skladni

Pipeline w Azure DevOps definiujemy w pliku YAML w repozytorium.
Domyslna nazwa to `azure-pipelines.yml` w katalogu glownym.

**Hierarchia elementow:**
```
Pipeline
 ├── trigger       - kiedy uruchamiac
 ├── pool          - na jakim agencie
 ├── variables     - zmienne
 └── stages        - etapy (opcjonalne)
      └── jobs     - zadania
           └── steps - kroki
                ├── script  - komenda powloki
                ├── task    - wbudowane zadanie ADO
                └── checkout- pobranie kodu
```

**Najwazniejsze wbudowane zmienne:**

| Zmienna                    | Wartosc                        |
|----------------------------|--------------------------------|
| `Build.SourceBranch`       | `refs/heads/main`              |
| `Build.SourceBranchName`   | `main`                         |
| `Build.BuildId`            | unikatowe ID buildu            |
| `Build.BuildNumber`        | numer w formacie YYYYMMDD.N    |
| `System.DefaultWorkingDirectory` | katalog z kodem         |
| `Agent.OS`                 | `Linux`, `Windows_NT`, `Darwin`|

> 💡 Plik YAML traktujemy jak kod - trafia do repozytorium,
> podlega review (PR), ma historie zmian (git blame).
> To fundamentalna roznica w stosunku do konfiguracji w UI.

In [ ]:
import yaml


# Podstawowy pipeline - trigger na main i develop
BASIC_PIPELINE = """
trigger:
  branches:
    include:
      - main
      - develop
      - feature/*

pool:
  vmImage: ubuntu-latest

variables:
  pythonVersion: '3.11'

jobs:
  - job: build
    displayName: 'Build and validate'
    steps:
      - checkout: self

      - task: UsePythonVersion@0
        displayName: 'Setup Python $(pythonVersion)'
        inputs:
          versionSpec: '$(pythonVersion)'

      - script: pip install -r requirements.txt
        displayName: 'Install dependencies'

      - script: python -m py_compile src/**/*.py
        displayName: 'Syntax check'
"""

# Parsujemy YAML i pokazujemy strukture
pipeline = yaml.safe_load(BASIC_PIPELINE)
print("Trigger branches:", pipeline["trigger"]["branches"]["include"])
print("Pool:            ", pipeline["pool"]["vmImage"])
print("Python version:  ", pipeline["variables"]["pythonVersion"])
steps = pipeline["jobs"][0]["steps"]
print(f"Liczba krokow:    {len(steps)}")
for step in steps:
    label = step.get("displayName") or list(step.keys())[0]
    print(f"  - {label}")

In [ ]:
# Budowanie pipeline'u jako slownik Pythona
def build_pipeline(python_version="3.11", branches=None):
    """Generate Azure Pipelines YAML config as Python dict."""
    if branches is None:
        branches = ["main", "develop"]
    return {
        "trigger": {"branches": {"include": branches}},
        "pool": {"vmImage": "ubuntu-latest"},
        "variables": {"pythonVersion": python_version},
        "jobs": [
            {
                "job": "build",
                "displayName": "Build",
                "steps": [
                    {"checkout": "self"},
                    {
                        "task": "UsePythonVersion@0",
                        "inputs": {"versionSpec": "$(pythonVersion)"},
                    },
                    {"script": "pip install -r requirements.txt",
                     "displayName": "Install dependencies"},
                ],
            }
        ],
    }


config = build_pipeline(python_version="3.12", branches=["main", "release/*"])
print(yaml.dump(config, default_flow_style=False, sort_keys=False))

---

### 🐍 Cwiczenia - YAML pipeline

**1.** Wczytaj `BASIC_PIPELINE` i dodaj do listy krokow
nowy krok `script` z komenda `flake8 src/` i `displayName`
`'Lint check'`. Wypisz zmieniony pipeline jako YAML.

**2.** Uzupelnij funkcje `add_variable(pipeline_dict, key, value)`,
ktora dodaje zmienną do slownika `variables` i zwraca
zaktualizowany slownik.

**3.** Napisz funkcje `validate_pipeline(yaml_str)`, ktora sprawdza
czy YAML pipeline zawiera wymagane klucze: `trigger`, `pool`,
`jobs`. Zwraca liste brakujacych kluczy. *(Trudniejsze)*

In [ ]:
# Cwiczenie 1 - dodanie kroku lint do pipeline'u
pipeline = yaml.safe_load(BASIC_PIPELINE)
lint_step = {
    "script": ...,
    "displayName": ...,
}
pipeline["jobs"][0]["steps"].append(lint_step)

print(yaml.dump(pipeline, default_flow_style=False, sort_keys=False))

In [ ]:
# Cwiczenie 2 - add_variable
def add_variable(pipeline_dict, key, value):
    ...


pipeline = yaml.safe_load(BASIC_PIPELINE)
pipeline = add_variable(pipeline, "testDir", "tests/")
pipeline = add_variable(pipeline, "coverageMin", "80")
print("Variables:", pipeline["variables"])

In [ ]:
# Cwiczenie 3 - validate_pipeline *(Trudniejsze)*
REQUIRED_KEYS = ["trigger", "pool", "jobs"]


def validate_pipeline(yaml_str):
    """Return list of missing required top-level keys."""
    ...


BROKEN_PIPELINE = """
pool:
  vmImage: ubuntu-latest
jobs:
  - job: build
    steps:
      - script: echo hello
"""
missing = validate_pipeline(BROKEN_PIPELINE)
print(f"Brakujace klucze: {missing}")  # ['trigger']

missing_ok = validate_pipeline(BASIC_PIPELINE)
print(f"Poprawny pipeline: {missing_ok}")  # []

---

## 3. 🔹 Pytest w pipeline - uruchamianie i raportowanie

Uruchamianie testow Pytest w Azure Pipelines wymaga trzech elementow:
1. **Instalacja zaleznosci** - `pip install -r requirements-test.txt`
2. **Uruchomienie Pytest** - z flagami `--junitxml` i `--html`
3. **Publikowanie wynikow** - task `PublishTestResults@2` i
   `PublishPipelineArtifact@1`

**Flagi Pytest wazne dla CI:**

| Flaga                          | Opis                                    |
|--------------------------------|-----------------------------------------|
| `--junitxml=report.xml`        | wyniki w formacie JUnit (ADO, Jenkins)  |
| `--html=report.html`           | raport HTML (pytest-html)               |
| `--self-contained-html`        | HTML bez zewnetrznych zasobow           |
| `-v`                           | verbose - widoczne nazwy testow         |
| `--tb=short`                   | skrocone tracebacki (czytelne w logach) |
| `--cov=src --cov-report=xml`   | pokrycie kodu (pytest-cov)              |
| `-m 'not slow'`                | pomijanie testow oznaczonych `@slow`    |

**Task PublishTestResults@2:**
```yaml
- task: PublishTestResults@2
  condition: always()   # publikuj nawet gdy testy failuja
  inputs:
    testResultsFormat: JUnit
    testResultsFiles: 'reports/test-results.xml'
    failTaskOnFailedTests: true
    testRunTitle: 'Pytest - $(Build.SourceBranchName)'
```

> 💡 `condition: always()` jest kluczowe - bez tego wyniki
> nie zostana opublikowane gdy testy nie przejda. Chcemy
> zawsze widziec ktore testy failuja, nie tylko czy failuja.

In [ ]:
# Kompletny pipeline testowy dla Pytest
PYTEST_PIPELINE = """
trigger:
  branches:
    include:
      - main
      - develop
      - feature/*
  paths:
    exclude:
      - '*.md'
      - docs/

pool:
  vmImage: ubuntu-latest

variables:
  pythonVersion: '3.11'
  reportsDir: '$(System.DefaultWorkingDirectory)/reports'

jobs:
  - job: pytest
    displayName: 'Run Pytest'
    steps:
      - checkout: self

      - task: UsePythonVersion@0
        displayName: 'Setup Python $(pythonVersion)'
        inputs:
          versionSpec: '$(pythonVersion)'

      - script: |
          pip install --upgrade pip
          pip install -r requirements-test.txt
        displayName: 'Install test dependencies'

      - script: mkdir -p $(reportsDir)
        displayName: 'Create reports directory'

      - script: |
          pytest tests/ \\
            --junitxml=$(reportsDir)/test-results.xml \\
            --html=$(reportsDir)/test-report.html \\
            --self-contained-html \\
            --cov=src \\
            --cov-report=xml:$(reportsDir)/coverage.xml \\
            --tb=short \\
            -v
        displayName: 'Run Pytest'

      - task: PublishTestResults@2
        condition: always()
        displayName: 'Publish test results'
        inputs:
          testResultsFormat: JUnit
          testResultsFiles: '$(reportsDir)/test-results.xml'
          failTaskOnFailedTests: true
          testRunTitle: 'Pytest - $(Build.SourceBranchName)'

      - task: PublishCodeCoverageResults@2
        condition: always()
        displayName: 'Publish coverage'
        inputs:
          summaryFileLocation: '$(reportsDir)/coverage.xml'

      - task: PublishPipelineArtifact@1
        condition: always()
        displayName: 'Upload HTML report'
        inputs:
          targetPath: '$(reportsDir)'
          artifact: test-reports
"""

pipeline = yaml.safe_load(PYTEST_PIPELINE)
steps = pipeline["jobs"][0]["steps"]
print(f"Liczba krokow: {len(steps)}")
for step in steps:
    name = step.get("displayName") or step.get("task") or list(step.keys())[0]
    cond = step.get("condition", "succeeded()")
    print(f"  [{cond:<12}] {name}")

---

### 🐍 Cwiczenia - Pytest w pipeline

**1.** Wczytaj `PYTEST_PIPELINE` i wypisz komende `script`
z kroku o `displayName` rownym `'Run Pytest'`.

**2.** Dodaj do `PYTEST_PIPELINE` krok uruchamiajacy `flake8`
przed krokiem z Pytest. Krok powinien miec
`displayName: 'Lint - flake8'` i komende `flake8 src/ tests/`.

**3.** Napisz funkcje `count_publish_tasks(yaml_str)`, ktora
zlicza ile krokow ma `condition: always()` w pipeline.
*(Trudniejsze)*

In [ ]:
# Cwiczenie 1 - komenda kroku 'Run Pytest'
pipeline = yaml.safe_load(PYTEST_PIPELINE)
steps = pipeline["jobs"][0]["steps"]

pytest_step = ...
print(pytest_step["script"])

In [ ]:
# Cwiczenie 2 - dodanie kroku lint przed pytest
# hint: znajdz indeks kroku 'Run Pytest' i wstaw przed nim
pipeline = yaml.safe_load(PYTEST_PIPELINE)
steps = pipeline["jobs"][0]["steps"]

lint_step = {
    "script": ...,
    "displayName": ...,
}

pytest_index = ...
steps.insert(pytest_index, lint_step)

for step in steps:
    name = step.get("displayName") or list(step.keys())[0]
    print(f"  - {name}")

In [ ]:
# Cwiczenie 3 - count_publish_tasks *(Trudniejsze)*
def count_always_steps(yaml_str):
    """Count pipeline steps with condition: always()."""
    ...


print(count_always_steps(PYTEST_PIPELINE))    # 3
print(count_always_steps(BASIC_PIPELINE))     # 0

---

## 4. 🔹 Multi-stage pipeline dla QA

Multi-stage pipeline rozdziela CI od CD na wyrazne etapy (stages).
Kazdy stage moze miec inne agenty, warunki i zatwierdzenia.

**Typowy przeplyw QA:**
```
test  --->  deploy  --->  smoke-test
 |              |               |
 | unit +       | deploy do     | testy e2e
 | integration  | staging       | po deploymencie
 | tests        |               |
 |              |               |
 v              v               v
fail? stop   fail? rollback  fail? alert
```

**Kluczowe elementy:**

| Element               | Opis                                           |
|-----------------------|------------------------------------------------|
| `dependsOn`           | stage uruchamia sie po innym                   |
| `condition: succeeded()` | uruchom tylko gdy poprzedni przeszedl       |
| `deployment`          | job z historia wdrozen i approval gates        |
| `environment`         | logiczne srodowisko (staging, prod) w ADO UI   |
| `approvals`           | manualne zatwierdzenie przed deployem do prod  |

> 💡 Approval gates konfigurujemy w ADO UI: Pipelines → Environments
> → [Environment] → Approvals and checks. Nie sa definiowane w YAML,
> ale YAML odwoluje sie do srodowiska przez `environment: staging`.

In [ ]:
MULTISTAGE_PIPELINE = """
trigger:
  - main

variables:
  pythonVersion: '3.11'

stages:

  - stage: test
    displayName: 'Test'
    jobs:
      - job: unit
        displayName: 'Unit tests'
        pool:
          vmImage: ubuntu-latest
        steps:
          - checkout: self
          - task: UsePythonVersion@0
            inputs:
              versionSpec: '$(pythonVersion)'
          - script: pip install -r requirements-test.txt
            displayName: 'Install deps'
          - script: |
              pytest tests/unit/ \\
                --junitxml=unit-results.xml \\
                --tb=short -v
            displayName: 'Run unit tests'
          - task: PublishTestResults@2
            condition: always()
            inputs:
              testResultsFormat: JUnit
              testResultsFiles: unit-results.xml
              testRunTitle: 'Unit - $(Build.BuildNumber)'
              failTaskOnFailedTests: true

      - job: integration
        displayName: 'Integration tests'
        pool:
          vmImage: ubuntu-latest
        steps:
          - checkout: self
          - task: UsePythonVersion@0
            inputs:
              versionSpec: '$(pythonVersion)'
          - script: pip install -r requirements-test.txt
          - script: |
              pytest tests/integration/ \\
                --junitxml=integration-results.xml \\
                --tb=short -v
            displayName: 'Run integration tests'
          - task: PublishTestResults@2
            condition: always()
            inputs:
              testResultsFormat: JUnit
              testResultsFiles: integration-results.xml
              testRunTitle: 'Integration - $(Build.BuildNumber)'
              failTaskOnFailedTests: true

  - stage: deploy
    displayName: 'Deploy to Staging'
    dependsOn: test
    condition: succeeded()
    jobs:
      - deployment: deploy_staging
        displayName: 'Deploy to staging'
        pool:
          vmImage: ubuntu-latest
        environment: staging
        strategy:
          runOnce:
            deploy:
              steps:
                - script: echo "Deploying build $(Build.BuildId) to staging"
                  displayName: 'Deploy'

  - stage: smoke
    displayName: 'Smoke Tests'
    dependsOn: deploy
    condition: succeeded()
    jobs:
      - job: smoke_tests
        displayName: 'Run smoke tests'
        pool:
          vmImage: ubuntu-latest
        steps:
          - checkout: self
          - task: UsePythonVersion@0
            inputs:
              versionSpec: '$(pythonVersion)'
          - script: pip install -r requirements-test.txt
          - script: |
              pytest tests/smoke/ \\
                --junitxml=smoke-results.xml \\
                -v
            displayName: 'Run smoke tests'
          - task: PublishTestResults@2
            condition: always()
            inputs:
              testResultsFormat: JUnit
              testResultsFiles: smoke-results.xml
              testRunTitle: 'Smoke - $(Build.BuildNumber)'
              failTaskOnFailedTests: true
"""

pipeline = yaml.safe_load(MULTISTAGE_PIPELINE)
stages = pipeline["stages"]
print(f"Liczba stages: {len(stages)}")
for stage in stages:
    depends = stage.get("dependsOn", "(brak)")
    cond = stage.get("condition", "(domyslny)")
    print(f"  {stage['stage']:<12} dependsOn={depends}, condition={cond}")

---

### 🐍 Cwiczenia - multi-stage pipeline

**1.** Wczytaj `MULTISTAGE_PIPELINE` i wypisz dla kazdego stage:
nazwe, `dependsOn` i liczbe jobs.

**2.** Dodaj do `MULTISTAGE_PIPELINE` nowy stage `report`
po `smoke`, ktory uruchamia skrypt `python scripts/send_report.py`.
Stage powinien miec `dependsOn: smoke` i `condition: always()`.

**3.** Napisz funkcje `get_stage_order(yaml_str)`, ktora
zwraca liste nazw stages w kolejnosci wykonania (uwzgledniajac
`dependsOn`). *(Trudniejsze)*

In [ ]:
# Cwiczenie 1 - info o stages
pipeline = yaml.safe_load(MULTISTAGE_PIPELINE)

for stage in pipeline["stages"]:
    name = stage["stage"]
    depends = ...
    job_count = ...
    print(f"{name:<12} dependsOn={depends}, jobs={job_count}")

In [ ]:
# Cwiczenie 2 - dodanie stage 'report'
pipeline = yaml.safe_load(MULTISTAGE_PIPELINE)

report_stage = {
    "stage": "report",
    "displayName": "Send Report",
    "dependsOn": ...,
    "condition": ...,
    "jobs": [
        {
            "job": "send_report",
            "pool": {"vmImage": "ubuntu-latest"},
            "steps": [
                {
                    "script": ...,
                    "displayName": "Send report",
                }
            ],
        }
    ],
}

pipeline["stages"].append(report_stage)
print(f"Stages po dodaniu: {[s['stage'] for s in pipeline['stages']]}")

In [ ]:
# Cwiczenie 3 - get_stage_order *(Trudniejsze)*
# hint: stage bez dependsOn idzie pierwszy, potem te ktore
# dependsOn poprzednich
def get_stage_order(yaml_str):
    """Return stage names in execution order based on dependsOn."""
    pipeline = yaml.safe_load(yaml_str)
    stages = pipeline["stages"]
    ...


order = get_stage_order(MULTISTAGE_PIPELINE)
print("Kolejnosc stages:", order)  # ['test', 'deploy', 'smoke']

---

## 5. 🔹 Azure DevOps REST API w Pythonie

Azure DevOps udostepnia kompletne REST API do automatyzacji:
uruchamiania pipeline'ow, pobierania wynikow testow, zarzadzania
branch policies. To kluczowe narzedzie dla QA Automation Engineer.

**Uwierzytelnianie - PAT (Personal Access Token):**
PAT tworzymy w ADO: User Settings → Personal access tokens.
W zadaniach HTTP uzywamy Basic Auth z pustym username i PAT
jako haslem: `auth=('', PAT)`.

**Podstawowe endpointy:**

| Zasob                  | Endpoint                          | Metoda |
|------------------------|-----------------------------------|--------|
| Projekty               | `/_apis/projects`                 | GET    |
| Pipeline'y             | `/_apis/pipelines`                | GET    |
| Uruchomienie pipeline  | `/_apis/pipelines/{id}/runs`      | POST   |
| Historia buildow       | `/_apis/build/builds`             | GET    |
| Status buildu          | `/_apis/build/builds/{id}`        | GET    |
| Test runs              | `/_apis/test/runs`                | GET    |
| Wyniki test run        | `/_apis/test/runs/{id}/results`   | GET    |
| Branch policies        | `/_apis/policy/configurations`    | GET    |

> 💡 Wszystkie endpointy wymagaja parametru `?api-version=7.1`.
> Odpowiedzi maja klucz `value` (lista) lub bezposredni obiekt.
> Limit stronicowania to 100 elementow - uzywamy `$top` i `$skip`.

In [ ]:
# Lista buildow - ostatnie 10
builds_data = client.get("build/builds", **{"$top": 10})
builds = builds_data.get("value", [])
print(f"Buildy (ostatnie {len(builds)}):")
print(f"{'ID':<8} {'Status':<12} {'Result':<12} {'Branch':<30} Numer")
print("-" * 78)
for b in builds:
    print(
        f"{b['id']:<8} "
        f"{b['status']:<12} "
        f"{b.get('result', 'running'):<12} "
        f"{b['sourceBranch'].replace('refs/heads/', ''):<30} "
        f"{b['buildNumber']}"
    )

In [ ]:
# Uruchomienie pipeline'u przez API
def trigger_pipeline(client, pipeline_id, branch="main", variables=None):
    """Trigger an Azure Pipelines run and return run info."""
    body = {
        "resources": {
            "repositories": {
                "self": {"refName": f"refs/heads/{branch}"}
            }
        }
    }
    if variables:
        body["variables"] = {
            k: {"value": str(v)} for k, v in variables.items()
        }
    return client.post(f"pipelines/{pipeline_id}/runs", body)


# Przyklad - uzupelnij pipeline_id swoim ID
# run = trigger_pipeline(client, pipeline_id=1, branch="develop")
# print(f"Uruchomiono: Run #{run['id']} - {run['state']}")
# print(f"URL: {run['_links']['web']['href']}")
print("Odkomentuj powyzszy kod po podaniu wlasciwego pipeline_id")

In [ ]:
# Wyniki testow z ostatniego buildu
def get_latest_test_results(client, top=5):
    """Get test results from the most recent completed build."""
    builds = client.get(
        "build/builds",
        **{"$top": 1, "statusFilter": "completed", "resultFilter": "failed,succeeded"}
    )
    if not builds.get("value"):
        return None, []
    build = builds["value"][0]

    runs = client.get("test/runs", **{"buildId": build["id"]})
    if not runs.get("value"):
        return build, []

    run_id = runs["value"][0]["id"]
    results = client.get(f"test/runs/{run_id}/results", **{"$top": top})
    return build, results.get("value", [])


build, results = get_latest_test_results(client)
if build:
    print(f"Build: {build['buildNumber']} | {build.get('result', 'N/A')}")
    print(f"Wyniki testow ({len(results)}):")
    for r in results:
        outcome = r.get("outcome", "unknown")
        icon = "OK" if outcome == "Passed" else "FAIL"
        print(f"  [{icon}] {r['testCase']['name']}")
else:
    print("Brak buildow - uzupelnij ADO_ORG, ADO_PROJECT, ADO_PAT")

---

### 🐍 Cwiczenia - ADO REST API

**1.** Pobierz liste buildow i znajdz ostatni `failed` build.
Wypisz jego ID, numer i galaz.

**2.** Napisz funkcje `get_pipeline_by_name(client, name)`,
ktora zwraca slownik pipeline'u o podanej nazwie lub `None`.

**3.** Napisz funkcje `build_summary(client, build_id)`, ktora
zwraca slownik z polami: `build_number`, `result`, `branch`,
`passed`, `failed`, `skipped` (liczby testow). *(Trudniejsze)*

**4.** Napisz funkcje `wait_for_build(client, run_id, timeout=300)`,
ktora odpytuje API co 10 sekund az build sie zakonczy lub
minie timeout. Zwraca finalny stan. *(Trudniejsze)*

In [ ]:
# Cwiczenie 1 - ostatni failed build
builds_data = client.get(
    "build/builds",
    **{"$top": 50, "statusFilter": "completed"}
)
builds = builds_data.get("value", [])

# hint: filtruj po polu 'result' == 'failed'
failed = ...
last_failed = ...

if last_failed:
    branch = last_failed["sourceBranch"].replace("refs/heads/", "")
    print(f"Ostatni failed build:")
    print(f"  ID:     {last_failed['id']}")
    print(f"  Numer:  {last_failed['buildNumber']}")
    print(f"  Galaz:  {branch}")
else:
    print("Brak failed buildow")

In [ ]:
# Cwiczenie 2 - get_pipeline_by_name
def get_pipeline_by_name(client, name):
    """Return pipeline dict with given name or None."""
    ...


# hint: uzyj client.get('pipelines') i filtruj po p['name']
p = get_pipeline_by_name(client, "azure-basics-ci")
if p:
    print(f"Znaleziono: {p['name']} (ID: {p['id']})")
else:
    print("Pipeline nie znaleziony")

In [ ]:
# Cwiczenie 3 - build_summary *(Trudniejsze)*
def build_summary(client, build_id):
    """
    Return summary dict:
    {build_number, result, branch, passed, failed, skipped}
    """
    build = client.get(f"build/builds/{build_id}")
    runs = client.get("test/runs", **{"buildId": build_id})

    passed = failed = skipped = 0
    for run in runs.get("value", []):
        passed += ...
        failed += ...
        skipped += ...

    return {
        "build_number": build["buildNumber"],
        "result": build.get("result", "unknown"),
        "branch": build["sourceBranch"].replace("refs/heads/", ""),
        "passed": passed,
        "failed": failed,
        "skipped": skipped,
    }


# Przyklad uzycia - podaj ID istniejacego buildu
# summary = build_summary(client, build_id=123)
# print(summary)

In [ ]:
# Cwiczenie 4 - wait_for_build *(Trudniejsze)*
import time


def wait_for_build(client, build_id, timeout=300, interval=10):
    """
    Poll build status until completed or timeout.
    Returns final status string.
    """
    start = time.time()
    while time.time() - start < timeout:
        build = client.get(f"build/builds/{build_id}")
        status = build["status"]
        result = build.get("result", "")
        elapsed = int(time.time() - start)
        print(f"  [{elapsed:>3}s] status={status} result={result or '-'}")

        if status == ...:
            return result

        time.sleep(interval)

    raise TimeoutError(f"Build {build_id} nie zakonczyl sie w {timeout}s")


# Przyklad uzycia po uruchomieniu pipeline'u
# run = trigger_pipeline(client, pipeline_id=1)
# result = wait_for_build(client, run['id'])
# print(f"Build zakonczony: {result}")